# GenSLM Classification Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramanathanlab/genslm/blob/main/examples/classification.ipynb)

This notebook demonstrates how to use GenSLM for genomic sequence classification tasks. We'll show how to:

1. Install and set up GenSLM
2. Load and prepare classification data
3. Train a classifier using pretrained GenSLM models
4. Make predictions on new sequences
5. Evaluate model performance

## Prerequisites

- Python 3.7+
- PyTorch
- Access to GenSLM model weights (download from Globus or use smaller models)

## Example Dataset

We'll use the EarthMicrobiome dataset to classify genomic sequences based on whether they are pathogens to bees.

## Setup and Installation

In [ ]:
# Install GenSLM (you may need to run this twice due to dependency conflicts)
!pip install git+https://github.com/ramanathanlab/genslm

In [ ]:
# If using Google Colab, mount your Google Drive to access model weights
from google.colab import drive
drive.mount('/content/gdrive')

## Import Libraries

In [ ]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path

# GenSLM imports
from genslm import (
    train_classifier,
    load_classifier,
    predict_from_csv,
    evaluate_classifier,
    GenSLMClassifier,
    ClassificationDataset
)

# Check if CUDA is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# For reproducibility
torch.manual_seed(42)
np.random.seed(42)

## Data Preparation

First, let's load and examine the EarthMicrobiome dataset.

In [ ]:
# Load the dataset
# Update this path to point to your EarthMicrobiome data
data_path = "path/to/your/EarthMicrobiome_with_embeddings.csv"

# For this example, let's create a smaller sample dataset
df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nTarget distribution:")
print(df['Pathogens_To_Bees'].value_counts())

In [ ]:
# For demonstration, let's work with a subset of the data
# Remove rows with missing sequences or labels
df_clean = df.dropna(subset=['BP', 'Pathogens_To_Bees'])

# Take a sample for faster training (optional)
sample_size = 1000  # Adjust based on your computational resources
if len(df_clean) > sample_size:
    df_sample = df_clean.sample(n=sample_size, random_state=42)
else:
    df_sample = df_clean.copy()

print(f"Working with {len(df_sample)} samples")
print(f"Target distribution in sample:")
print(df_sample['Pathogens_To_Bees'].value_counts())

# Save sample for training
sample_path = "earthmicrobiome_sample.csv"
df_sample.to_csv(sample_path, index=False)
print(f"\nSample saved to: {sample_path}")

## Examine Genomic Sequences

In [ ]:
# Examine sequence lengths
sequence_lengths = df_sample['BP'].str.len()

print(f"Sequence length statistics:")
print(f"  Mean: {sequence_lengths.mean():.1f}")
print(f"  Median: {sequence_lengths.median():.1f}")
print(f"  Min: {sequence_lengths.min()}")
print(f"  Max: {sequence_lengths.max()}")
print(f"  Std: {sequence_lengths.std():.1f}")

# Show some example sequences
print(f"\nExample sequences:")
for i in range(3):
    seq = df_sample.iloc[i]['BP']
    label = df_sample.iloc[i]['Pathogens_To_Bees']
    print(f"\nSample {i+1} (Label: {label}):")
    print(f"Length: {len(seq)}")
    print(f"First 100 bp: {seq[:100]}...")

## Training a GenSLM Classifier

Now let's train a classifier using the high-level API. This will use a pretrained GenSLM model as the backbone and add a classification head.

In [ ]:
# Set up model cache directory (where your GenSLM weights are stored)
model_cache_dir = "/content/gdrive/MyDrive"  # Update this path

# Training parameters
training_config = {
    "data_path": sample_path,
    "sequence_col": "BP",
    "target_col": "Pathogens_To_Bees", 
    "model_id": "genslm_25M_patric",  # Use smaller model for demo
    "model_cache_dir": model_cache_dir,
    "output_dir": "./bee_pathogen_classifier",
    "batch_size": 4,  # Small batch size for demo
    "max_epochs": 5,  # Few epochs for demo
    "learning_rate": 1e-4,
    "hidden_sizes": [256, 128],  # Simple MLP head
    "dropout": 0.1,
    "freeze_backbone": True,  # Freeze GenSLM weights
    "pooling_strategy": "mean",
    "patience": 2,
    "random_seed": 42
}

print("🚀 Starting training...")
print(f"Configuration: {training_config}\n")

# Train the classifier
results = train_classifier(**training_config)

print("\n✅ Training completed!")
print(f"Best model saved to: {results['best_model_path']}")

## Examining Training Results

In [ ]:
# Show training results
print("📊 Training Results:")
print(f"  Number of classes: {results['num_classes']}")
print(f"  Class names: {results['class_names']}")
print(f"  Training samples: {results['train_size']}")
print(f"  Validation samples: {results['val_size']}")
print(f"  Test samples: {results['test_size']}")

if results['test_results']:
    test_acc = results['test_results'].get('test/acc', 'N/A')
    test_f1 = results['test_results'].get('test/f1', 'N/A')
    print(f"\n🎯 Test Performance:")
    print(f"  Accuracy: {test_acc:.4f}" if test_acc != 'N/A' else f"  Accuracy: {test_acc}")
    print(f"  F1 Score: {test_f1:.4f}" if test_f1 != 'N/A' else f"  F1 Score: {test_f1}")

# Show hyperparameters used
print(f"\n⚙️ Hyperparameters:")
for key, value in results['hyperparameters'].items():
    print(f"  {key}: {value}")

## Making Predictions

Now let's use our trained model to make predictions on new sequences.

In [ ]:
# Load the trained model
best_model_path = results['best_model_path']
classifier = load_classifier(best_model_path, model_cache_dir)

print(f"✅ Loaded trained classifier from: {best_model_path}")

# Create some test sequences (or use a separate test file)
test_sequences = [
    "ATGAAAGTAACCGTTGTTGGAGCAGGTGCAGTTGGTGCAAGTTGCGCAGAATATATTGCA",
    "GGCTCAGGACGAACGCTGGCGGCGTGGATTAGGCATGCAAGTCGAGCGACGGACCTTCGGG",
    "TGTGACCCAGCGACGCCGCGTGAAGGATGAAGGCCCTCTGGGTTGTAAACTTCTTTTACG"
]

# Make predictions
with torch.no_grad():
    probabilities = classifier.predict(test_sequences, batch_size=8)
    predictions = torch.argmax(probabilities, dim=1)

# Show results
print("\n🔮 Predictions on test sequences:")
for i, (seq, pred, probs) in enumerate(zip(test_sequences, predictions, probabilities)):
    confidence = torch.max(probs).item()
    print(f"\nSequence {i+1}:")
    print(f"  Length: {len(seq)}")
    print(f"  Sequence: {seq[:50]}...")
    print(f"  Predicted class: {pred.item()}")
    print(f"  Confidence: {confidence:.4f}")
    print(f"  All probabilities: {probs.numpy()}")

## Batch Predictions from CSV

You can also make predictions on entire CSV files using the high-level API.

In [ ]:
# Create a small test CSV
test_data = {
    'sequence_id': ['test_1', 'test_2', 'test_3'],
    'BP': test_sequences,
    'source': ['synthetic', 'synthetic', 'synthetic']
}
test_df = pd.DataFrame(test_data)
test_csv_path = "test_sequences.csv"
test_df.to_csv(test_csv_path, index=False)

print(f"Created test CSV: {test_csv_path}")

# Make predictions using the CSV API
predictions_df = predict_from_csv(
    model_path=best_model_path,
    data_path=test_csv_path,
    sequence_col="BP",
    output_path="predictions.csv",
    batch_size=8,
    model_cache_dir=model_cache_dir
)

print("\n📊 Predictions DataFrame:")
print(predictions_df[['sequence_id', 'predicted_class', 'prediction_confidence']].head())

## Model Evaluation

Let's evaluate our model more thoroughly using test data.

In [ ]:
# Evaluate on the original test data (if you have ground truth labels)
# For this example, we'll use a subset of our training data
eval_data = df_sample.sample(n=100, random_state=42)  # Small subset for demo
eval_csv_path = "eval_data.csv"
eval_data.to_csv(eval_csv_path, index=False)

# Evaluate the model
eval_results = evaluate_classifier(
    model_path=best_model_path,
    data_path=eval_csv_path,
    sequence_col="BP",
    target_col="Pathogens_To_Bees",
    model_cache_dir=model_cache_dir,
    batch_size=8
)

print("\n📈 Detailed Evaluation Results:")
print(f"Accuracy: {eval_results['accuracy']:.4f}")
print(f"Macro F1: {eval_results['macro_f1']:.4f}")
print(f"Weighted F1: {eval_results['weighted_f1']:.4f}")

# Show confusion matrix
print("\n📊 Confusion Matrix:")
cm = np.array(eval_results['confusion_matrix'])
print(cm)

## Advanced Usage

For more advanced usage, you can work directly with the PyTorch Lightning model.

In [ ]:
# Load model for advanced usage
from genslm import GenSLM

# Access the backbone model directly
backbone = classifier.backbone
print(f"Backbone model: {backbone.model_info}")
print(f"Sequence length: {backbone.seq_length}")
print(f"Vocabulary size: {len(backbone.tokenizer)}")

# Access the classification head
print(f"\nClassification head: {classifier.classifier}")
print(f"Number of parameters: {sum(p.numel() for p in classifier.classifier.parameters())}")

# Model hyperparameters
print(f"\nModel hyperparameters:")
for key, value in classifier.hparams.items():
    print(f"  {key}: {value}")

## Tips for Better Performance

1. **Data Quality**: Ensure your genomic sequences are high quality and properly formatted
2. **Model Size**: Use larger GenSLM models (250M, 2.5B, 25B) for better performance if computational resources allow
3. **Training Time**: Increase `max_epochs` and reduce `batch_size` if you have limited GPU memory
4. **Fine-tuning**: Set `freeze_backbone=False` to fine-tune the entire model (requires more resources)
5. **Class Imbalance**: The model automatically handles class imbalance with class weights
6. **Sequence Length**: GenSLM models have a maximum sequence length (typically 2048 tokens)

## Command Line Usage

You can also use GenSLM classification from the command line:

```bash
# Train a classifier
genslm-train-classifier data.csv \
    --sequence-col BP \
    --target-col Pathogens_To_Bees \
    --model-id genslm_25M_patric \
    --output-dir ./results \
    --batch-size 8 \
    --max-epochs 10

# Make predictions
genslm-predict-classifier predict \
    ./results/checkpoints/best_model.ckpt \
    new_data.csv \
    --sequence-col BP \
    --output-path predictions.csv

# Evaluate a model
genslm-predict-classifier evaluate \
    ./results/checkpoints/best_model.ckpt \
    test_data.csv \
    --sequence-col BP \
    --target-col Pathogens_To_Bees
```

## Conclusion

This notebook showed how to use GenSLM for genomic sequence classification. The key benefits of this approach are:

- **Pretrained Representations**: Leverages large-scale genomic language models
- **Easy to Use**: High-level API for common tasks
- **Flexible**: Configurable architecture and training parameters
- **Scalable**: Works with large datasets and can use multiple GPUs

For more information, see the [GenSLM documentation](https://github.com/ramanathanlab/genslm).